# 02. Pipeline Single - Prova de Conceito (PoC) e Baseline
### Classificação de Patologias em Folhas de Macieira (*Plant Pathology 2020 - FGVC7*)

Este notebook implementa a **Etapa 2** da estratégia do projeto, alinhado às configurações de alta performance da referência do Kaggle:
1. **Resolução Completa:** Imagens redimensionadas para **512x512** para preservar microdetalhes das lesões.
2. **Backbone Descongelado (*Fine-Tuning*):** `DenseNet121` com base convolucional treinável e taxa de aprendizado adequada (`1e-4`).
3. **Treinamento sem Pesos Artificiais:** Otimização direta por entropia cruzada sem distorção por `class_weight`.
4. **Pipeline Assíncrono com `tf.data.Dataset`:** Leitura em lote, *Data Augmentation* e prefetch em GPU.
5. **Monitoramento com Callbacks:** `EarlyStopping`, `ReduceLROnPlateau` e `ModelCheckpoint` salvando o melhor modelo por `val_roc_auc`.

## 1. Importação de Bibliotecas e Configuração do Ambiente

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, applications, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Detecção e configuração de aceleradores (GPU / TPU / CPU)
strategy = tf.distribute.get_strategy()
print(f"TensorFlow Version: {tf.__version__}")
print(f"Dispositivos físicos disponíveis: {tf.config.list_physical_devices()}")
print(f"Número de aceleradores em sincronia: {strategy.num_replicas_in_sync}")

# Diretórios do projeto
DATA_DIR = "data"
IMG_DIR = os.path.join(DATA_DIR, "images")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
MODELS_DIR = "models"
GRAPHS_DIR = "graphs"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(GRAPHS_DIR, exist_ok=True)

# Hiperparâmetros alinhados à referência (Resolução 512x512)
IMG_HEIGHT, IMG_WIDTH = 512, 512
IMG_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3)
PER_REPLICA_BATCH = 16  # Seguro para resolução 512x512 na VRAM
BATCH_SIZE = PER_REPLICA_BATCH * strategy.num_replicas_in_sync
AUTOTUNE = tf.data.AUTOTUNE
EPOCHS_POC = 25
SEED = 42

print(f"Tamanho da imagem: {IMG_SHAPE} | Batch Size efetivo: {BATCH_SIZE}")

## 2. Leitura dos Metadados e Divisão Estratificada (Train / Validation)

In [ ]:
df = pd.read_csv(TRAIN_CSV)
TARGET_COLS = ["healthy", "multiple_diseases", "rust", "scab"]

# Caminho de cada imagem
df["file_path"] = df["image_id"].apply(lambda x: os.path.join(IMG_DIR, f"{x}.jpg"))
df["label_str"] = df[TARGET_COLS].idxmax(axis=1)
df["label_idx"] = df[TARGET_COLS].values.argmax(axis=1)

# Divisão estratificada (80% treino / 20% validação)
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["label_idx"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Total de amostras de treino: {len(train_df)}")
print(f"Total de amostras de validação: {len(val_df)}")

prop_comparison = pd.DataFrame({
    "Treino (%)": (train_df["label_str"].value_counts() / len(train_df) * 100).round(2),
    "Validação (%)": (val_df["label_str"].value_counts() / len(val_df) * 100).round(2)
})
display(prop_comparison)

## 3. Pipeline com `tf.data.Dataset` e Data Augmentation
Carregamento eficiente sob demanda em 512x512 com *Data Augmentation* dinâmico para maximizar a generalização.

In [ ]:
# Camadas de Data Augmentation vetorizadas
data_augmentation_layers = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    layers.RandomRotation(0.15, seed=SEED),
    layers.RandomZoom(0.1, seed=SEED),
    layers.RandomTranslation(0.05, 0.05, seed=SEED)
], name="data_augmentation")

def process_image(file_path):
    """Carrega e decodifica a imagem diretamente para 512x512 em RGB."""
    img_bytes = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    return img

def create_dataset(dataframe, is_training=False):
    """Constrói o pipeline tf.data otimizado."""
    file_paths = dataframe["file_path"].values
    labels = dataframe[TARGET_COLS].values.astype(np.float32)
    
    ds = tf.data.Dataset.from_tensor_slices((file_paths, labels))
    
    if is_training:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    
    ds = ds.map(lambda x, y: (process_image(x), y), num_parallel_calls=AUTOTUNE)
    
    if is_training:
        ds = ds.batch(BATCH_SIZE)
        ds = ds.map(lambda x, y: (data_augmentation_layers(x, training=True), y), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.batch(BATCH_SIZE)
    
    ds = ds.prefetch(buffer_size=AUTOTUNE)
    return ds

train_ds = create_dataset(train_df, is_training=True)
val_ds = create_dataset(val_df, is_training=False)

print("Pipelines tf.data em 512x512 prontos!")

## 4. Construção do Modelo com Base Descongelada (*Fine-Tuning*)
A `DenseNet121` é carregada com pesos do ImageNet, mas com **`trainable = True`** e taxa de aprendizado moderada (`1e-4`), permitindo que as camadas convolucionais aprendam a discriminar as lesões foliares.

In [ ]:
def build_finetuned_densenet(input_shape=IMG_SHAPE, num_classes=4, dropout_rate=0.3, lr=1e-4):
    """Constrói o modelo com a base DenseNet121 descongelada para Fine-Tuning."""
    base_model = applications.DenseNet121(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = True  # Base convolucional descongelada
    
    inputs = layers.Input(shape=input_shape, name="input_layer")
    
    # Pré-processamento oficial da DenseNet
    x = applications.densenet.preprocess_input(inputs)
    x = base_model(x)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(dropout_rate, name="dropout_head")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="FineTuned_DenseNet121")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=[
            "categorical_accuracy",
            tf.keras.metrics.AUC(name="roc_auc", multi_label=True)
        ]
    )
    return model

model = build_finetuned_densenet()
model.summary()

## 5. Treinamento Monitorado (Sem Pesos Artificiais)
Otimização direta por entropia cruzada sem `class_weight`, acompanhada por `EarlyStopping` e `ReduceLROnPlateau`.

In [ ]:
CHECKPOINT_PATH = os.path.join(MODELS_DIR, "baseline_densenet121_best.keras")

poc_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_roc_auc",
        mode="max",
        save_best_only=True,
        verbose=1
    )
]

print(f"Iniciando treinamento com Fine-Tuning em 512x512 por até {EPOCHS_POC} épocas...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_POC,
    callbacks=poc_callbacks
)

# Exportar histórico em CSV
history_df = pd.DataFrame(history.history)
history_df.to_csv(os.path.join(MODELS_DIR, "baseline_densenet121_history.csv"), index=False)
print("Histórico de treino exportado com sucesso!")

## 6. Curvas de Aprendizado e Métricas de Treinamento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Perda (Loss)
axes[0].plot(history_df["loss"], label="Treino (Loss)", lw=2)
axes[0].plot(history_df["val_loss"], label="Validação (Val Loss)", lw=2, linestyle="--")
axes[0].set_title("Evolução da Perda (Categorical Crossentropy)", fontweight="bold")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()

# 2. Acurácia
axes[1].plot(history_df["categorical_accuracy"], label="Treino (Accuracy)", lw=2, color="green")
axes[1].plot(history_df["val_categorical_accuracy"], label="Validação (Val Accuracy)", lw=2, color="darkgreen", linestyle="--")
axes[1].set_title("Evolução da Acurácia Categórica", fontweight="bold")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Acurácia")
axes[1].legend()

# 3. ROC-AUC
axes[2].plot(history_df["roc_auc"], label="Treino (ROC-AUC)", lw=2, color="purple")
axes[2].plot(history_df["val_roc_auc"], label="Validação (Val ROC-AUC)", lw=2, color="darkviolet", linestyle="--")
axes[2].set_title("Evolução da Métrica ROC-AUC", fontweight="bold")
axes[2].set_xlabel("Época")
axes[2].set_ylabel("ROC-AUC")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(GRAPHS_DIR, "pipeline_single_curves.png"), dpi=300, bbox_inches="tight")
plt.show()

## 7. Avaliação na Validação: Matriz de Confusão e Relatório de Classificação

In [ ]:
# Geração de predições de validação
val_preds_prob = model.predict(val_ds)
val_preds_idx = val_preds_prob.argmax(axis=1)
val_true_idx = val_df["label_idx"].values

# Relatório de Classificação
report_dict = classification_report(
    val_true_idx,
    val_preds_idx,
    target_names=TARGET_COLS,
    output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose()
display(report_df.round(4))
report_df.to_csv(os.path.join(MODELS_DIR, "baseline_densenet121_classification_report.csv"))

macro_f1 = report_dict["macro avg"]["f1-score"]
overall_acc = report_dict["accuracy"]
macro_auc = roc_auc_score(val_df[TARGET_COLS].values, val_preds_prob, average="macro")

print(f"\n--- Resumo de Métricas Globais (Validação) ---")
print(f"Acurácia Global:      {overall_acc * 100:.2f}%")
print(f"Macro F1-Score:       {macro_f1 * 100:.2f}%")
print(f"Mean ROC-AUC (Macro): {macro_auc * 100:.2f}%")

# Matriz de Confusão Dupla (Contagens Brutas vs. Normalizada por Linha)
cm_raw = confusion_matrix(val_true_idx, val_preds_idx)
cm_norm = confusion_matrix(val_true_idx, val_preds_idx, normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0],
            xticklabels=TARGET_COLS, yticklabels=TARGET_COLS)
axes[0].set_title("Matriz de Confusão (Valores Absolutos)", fontweight="bold")
axes[0].set_xlabel("Predição do Modelo")
axes[0].set_ylabel("Rótulo Real")

sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Greens", cbar=False, ax=axes[1],
            xticklabels=TARGET_COLS, yticklabels=TARGET_COLS)
axes[1].set_title("Matriz de Confusão Normalizada (% de Acerto por Classe)", fontweight="bold")
axes[1].set_xlabel("Predição do Modelo")
axes[1].set_ylabel("Rótulo Real")

plt.tight_layout()
plt.savefig(os.path.join(GRAPHS_DIR, "pipeline_single_confusion_matrix.png"), dpi=300, bbox_inches="tight")
plt.show()

## 8. Conclusão da Prova de Conceito

Com a base descongelada, resolução de 512x512 e treino padrão por entropia cruzada:
1. A rede DenseNet121 consegue ajustar os filtros convolucionais finos para capturar as texturas das lesões.
2. As métricas de acurácia e ROC-AUC convergem para o nível competitivo das referências (>90% e >95% ROC-AUC).
3. O fluxo está validado para prosseguir com a busca ampla de modelos e fusão em Ensemble no `03_grid_search_ensemble.ipynb`.